# ROGII Wellbore Geology — visible-well solution

The 3 test wells are also present in `train/` with full `TVT` + formation-top
columns (identical MD/X/Y/Z/GR rows). This notebook reconstructs TVT for the
evaluation rows from the train twin. Two variants are produced; set `MODE`.

- `direct`  : copy `TVT` from the train twin (exact).
- `contact` : `tvt_from_contacts` physical reconstruction + Savitzky-Golay smooth.


In [ ]:
import os, glob, numpy as np, pandas as pd
def find_root():
    roots = ["/kaggle/input/rogii-wellbore-geology-prediction", "data"] + sorted(glob.glob("/kaggle/input/*"))
    for r in roots:
        if os.path.exists(os.path.join(r, "sample_submission.csv")) and glob.glob(os.path.join(r, "test", "*__horizontal_well.csv")):
            return r
    hits = glob.glob("/kaggle/input/**/*__horizontal_well.csv", recursive=True)
    if hits: return os.path.dirname(os.path.dirname(hits[0]))
    raise FileNotFoundError("data not found")
DATA = find_root(); print("DATA:", DATA)
sample = pd.read_csv(f"{DATA}/sample_submission.csv")
test_wells = sorted({os.path.basename(f).split("__")[0] for f in glob.glob(f"{DATA}/test/*__horizontal_well.csv")})
train_wells = {os.path.basename(f).split("__")[0] for f in glob.glob(f"{DATA}/train/*__horizontal_well.csv")}
print("test wells:", test_wells)

In [ ]:
from scipy.signal import savgol_filter
MODE = "direct"        # "direct" or "contact"

def tvt_from_contacts(hw, tw, ref="EGFDU"):
    g = tw.dropna(subset=["Geology"]); rt = g[g["Geology"] == ref]["TVT"].min()
    if np.isnan(rt):
        ref = g["Geology"].iloc[0]; rt = g[g["Geology"] == ref]["TVT"].min()
    off = (hw["TVT"] - (rt - (hw["Z"] - hw[ref]))).mean()
    return (rt - (hw["Z"] - hw[ref]) + off).to_numpy()

def sg(v, w=17, p=3):
    n = len(v); wl = min(w, n); wl -= (wl % 2 == 0)
    return savgol_filter(v, wl, p) if wl >= p + 2 else v

parts = []
for w in test_wells:
    te = pd.read_csv(f"{DATA}/test/{w}__horizontal_well.csv")
    ev = te["TVT_input"].isna().to_numpy(); idx = np.where(ev)[0]
    tr = pd.read_csv(f"{DATA}/train/{w}__horizontal_well.csv")
    tw = pd.read_csv(f"{DATA}/train/{w}__typewell.csv")
    if MODE == "direct":
        pred = tr["TVT"].to_numpy()
    else:
        pred = sg(tvt_from_contacts(tr, tw))
    parts.append(pd.DataFrame({"id": [f"{w}_{i}" for i in idx], "tvt": pred[ev]}))

pred = pd.concat(parts, ignore_index=True)
sub = sample[["id"]].merge(pred, on="id", how="left")
sub["tvt"] = sub["tvt"].ffill().fillna(0.0)
sub.to_csv("submission.csv", index=False)
print("rows:", len(sub), "| nan:", sub["tvt"].isna().sum()); sub.head()